In [1]:
import torch
import os
import random
import numpy as np
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader

In [2]:
random_seed = 37
class EmbedDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir):
        self.emb_dir = os.path.join(data_dir, 'embeds')
        self.speakers = os.listdir(self.emb_dir)
        random.seed(random_seed)
        random.shuffle(self.speakers)
        self.train_info = []
        for spk in self.speakers:
            mel_ids = os.listdir(os.path.join(self.emb_dir, spk))
            self.train_info += [(i[:-10], spk) for i in mel_ids]

        print("Total number of training embeds is %d." % len(self.train_info))
        print("Total number of training speakers is %d." % len(self.speakers))
        random.seed(random_seed)
        random.shuffle(self.train_info)

    def get_data(self, audio_info):
        audio_id, spk = audio_info
        embed = self.get_embed(audio_id, spk)
        return embed

    def get_embed(self, audio_id, spk):
        embed_path = os.path.join(self.emb_dir, spk, audio_id + '_embed.npy')
        embed = np.load(embed_path)
        embed = torch.from_numpy(embed).float()
        return embed

    def __getitem__(self, index):
        embed = self.get_data(self.train_info[index])
        item = {'embed': embed}
        return item

    def __len__(self):
        return len(self.train_info)

In [3]:
'''
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.FC_input = nn.Linear(input_dim, hidden_dim)
        self.FC_input2 = nn.Linear(hidden_dim, hidden_dim)

        self.FC_mean = nn.Linear(hidden_dim, latent_dim)
        self.FC_var = nn.Linear(hidden_dim, latent_dim)

        self.LeakyReLU = nn.LeakyReLU(0.2)

        self.training = True

    def forward(self, x):
        h_ = self.LeakyReLU(self.FC_input(x))
        h_ = self.LeakyReLU(self.FC_input2(h_))

        mean = self.FC_mean(h_)
        # encoder produces mean and log of variance
        # (i.e., parateters of simple tractable normal distribution "q"
        log_var = self.FC_var(h_)
        return mean, log_var
'''

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.FC_input = nn.Linear(input_dim, hidden_dim)
        self.FC_mean = nn.Linear(hidden_dim, latent_dim)
        self.FC_var = nn.Linear(hidden_dim, latent_dim)
        self.LeakyReLU = nn.LeakyReLU(0.2)
        self.training = True

    def forward(self, x):
        h_ = self.LeakyReLU(self.FC_input(x))
        mean = self.FC_mean(h_)
        log_var = self.FC_var(h_)
        return mean, log_var

In [4]:
'''
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.FC_hidden = nn.Linear(latent_dim, hidden_dim)
        self.FC_hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.FC_output = nn.Linear(hidden_dim, output_dim)
        self.LeakyReLU = nn.LeakyReLU(0.2)

    def forward(self, x):
        h = self.LeakyReLU(self.FC_hidden(x))
        h = self.LeakyReLU(self.FC_hidden2(h))
        x_hat = torch.tanh(self.FC_output(h))
        return x_hat
'''    
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.FC_hidden = nn.Linear(latent_dim, hidden_dim)
        self.FC_output = nn.Linear(hidden_dim, output_dim)
        self.LeakyReLU = nn.LeakyReLU(0.2)

    def forward(self, x):
        h = self.LeakyReLU(self.FC_hidden(x))
        # x_hat = torch.tanh(self.FC_output(h))
        x_hat = self.FC_output(h)
        return x_hat

In [5]:
class VAE(nn.Module):
    def __init__(self, x_dim, hidden_dim, latent_dim, DEVICE):
        super(VAE, self).__init__()
        self.Encoder = Encoder(input_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)
        self.Decoder = Decoder(latent_dim=latent_dim, hidden_dim = hidden_dim, output_dim = x_dim)
        self.DEVICE = DEVICE

    def reparameterization(self, mean, var):
        epsilon = torch.randn_like(var).to(self.DEVICE)  # sampling epsilon
        z = mean + var*epsilon  # reparameterization trick
        return z

    def forward(self, x):
        mean, log_var = self.Encoder(x)
        # takes exponential function (log var -> var)
        z = self.reparameterization(mean, torch.exp(0.5 * log_var))
        x_hat = self.Decoder(z)

        return x_hat, mean, log_var

In [6]:
cuda = True
DEVICE = torch.device("cuda" if cuda else "cpu")

# Model Hyperparameters
batch_size = 128

x_dim = 256
hidden_dim = 384
latent_dim = 64

lr = 1e-3

epochs = 50

In [7]:
model = VAE(x_dim, hidden_dim, latent_dim, DEVICE).to(DEVICE)

In [ ]:
###for rebuttal###

In [8]:
model.load_state_dict(torch.load('psg_mylossf_50.pt'))

<All keys matched successfully>

In [41]:
with torch.no_grad():
    noise = torch.randn(1, 64).to(DEVICE)
    emb1 = model.Decoder(noise)
emb1.shape

torch.Size([1, 256])

In [42]:
cos = nn.CosineSimilarity(dim=1, eps=1e-6)

In [45]:
cos(emb1,emb1)

tensor([1.], device='cuda:0')

In [46]:
all_sims=[]
train_set = EmbedDataset('dataset/libritts_subset/') 
train_loader = DataLoader(train_set, batch_size=1, num_workers=8, drop_last=True)
for batch_idx, dict in enumerate(train_loader):
    x = dict['embed']
    x = x.to(DEVICE)
    sim = cos(emb1,x)
    all_sims.append(sim.item())
    # print(sim.item())
    # break
# all_sims

Total number of training embeds is 46440.
Total number of training speakers is 2322.


In [23]:
import numpy as np

In [47]:
len(all_sims),np.mean(all_sims),np.var(all_sims)

(46440, 0.599210743156232, 0.005449444695111803)

In [48]:
sims=1-np.array(all_sims)

In [49]:
np.mean(sims),np.var(sims),np.std(sims)

(0.4007892568437679, 0.005449444695111802, 0.07382035420608467)

In [50]:
all_dists=[]
train_set = EmbedDataset('dataset/libritts_subset/') 
train_loader = DataLoader(train_set, batch_size=1, num_workers=8, drop_last=True)
for batch_idx, dict in enumerate(train_loader):
    x = dict['embed']
    x = x.to(DEVICE)
    with torch.no_grad():
        noise = torch.randn(1, 64).to(DEVICE)
        emb = model.Decoder(noise)
    sim1 = cos(emb,x)
    all_dists.append(1-sim1.item())

Total number of training embeds is 46440.
Total number of training speakers is 2322.


In [51]:
len(all_dists),np.mean(all_dists),np.var(all_dists),np.std(all_dists)

(46440, 0.3948887280336903, 0.006076646086556244, 0.07795284527556542)

In [54]:
all_dists=[]
vctk = EmbedDataset('dataset/vctk/') 
vctk_loader = DataLoader(vctk, batch_size=1, num_workers=8, drop_last=True)
for batch_idx, dict in enumerate(vctk_loader):
    x = dict['embed']
    x = x.to(DEVICE)
    with torch.no_grad():
        noise = torch.randn(1, 64).to(DEVICE)
        emb = model.Decoder(noise)
    sim1 = cos(emb,x)
    all_dists.append(1-sim1.item())
len(all_dists),np.mean(all_dists),np.var(all_dists),np.std(all_dists)

Total number of training embeds is 44242.
Total number of training speakers is 109.


(44242, 0.4259370556394649, 0.005037377198486195, 0.07097448272785224)

In [58]:
with torch.no_grad():
    noise = torch.randn(1, 64).to(DEVICE)
    emb = model.Decoder(noise)
all_dists=[]
vctk = EmbedDataset('dataset/vctk/') 
vctk_loader = DataLoader(vctk, batch_size=1, num_workers=8, drop_last=True)
for batch_idx, dict in enumerate(vctk_loader):
    x = dict['embed']
    x = x.to(DEVICE)
    sim1 = cos(emb,x)
    all_dists.append(1-sim1.item())
len(all_dists),np.mean(all_dists),np.var(all_dists),np.std(all_dists)

Total number of training embeds is 44242.
Total number of training speakers is 109.


(44242, 0.36332197565628993, 0.0025591844090932137, 0.05058838215532509)

In [ ]:
### ###

In [8]:
cos = nn.CosineSimilarity(dim=1, eps=1e-6)
mse = nn.MSELoss(reduction='sum')
l1 = nn.L1Loss(reduction='sum')
sml1 = nn.SmoothL1Loss(reduction='sum')

def my_loss_fun(x, x_hat, mean, log_var):
    cos_distance_loss = 200*(1-cos(x, x_hat)).sum() #or 100?  不行，cos_sim太低
    sml1_loss = sml1(x, x_hat)
    reconstruction_loss = cos_distance_loss + sml1_loss
    KLD = - 0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reconstruction_loss, KLD

def loss_function2(x, x_hat, mean, log_var):
    # cosine similarity loss
    cos_distance_loss = 100*(1-cos(x, x_hat)).sum()
    mse_loss = mse(x, x_hat)
    reconstruction_loss = cos_distance_loss + mse_loss
    KLD = - 0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reconstruction_loss, KLD

def loss_function1(x, x_hat, mean, log_var): #√
    # cosine similarity loss
    cos_distance_loss = 200*(1-cos(x, x_hat)).sum()
    l1_loss = l1(x, x_hat)
    reconstruction_loss = cos_distance_loss + l1_loss
    KLD = - 0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reconstruction_loss, KLD

def loss_function(x, x_hat, mean, log_var):
    mse_loss = mse(x, x_hat)
    reconstruction_loss = 10 * mse_loss
    KLD = - 0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reconstruction_loss, KLD

optimizer = Adam(model.parameters(), lr=lr)

In [9]:
train_set = EmbedDataset('dataset/libritts_subset/') 
train_loader = DataLoader(train_set, batch_size=batch_size, num_workers=8, drop_last=True)

Total number of training embeds is 46440.
Total number of training speakers is 2322.


In [10]:
print("Start training VAE...")
model.train()

for epoch in range(epochs):
    overall_loss, overall_reconst, overall_KLD = 0, 0, 0
    for batch_idx, dict in enumerate(train_loader):
        x = dict['embed']
        x = x.to(DEVICE)

        optimizer.zero_grad()

        x_hat, mean, log_var = model(x)
        reconst_loss, KLD = my_loss_fun(x, x_hat, mean, log_var)
        loss = reconst_loss + KLD
        overall_loss += loss.item()
        overall_reconst += reconst_loss.item()
        overall_KLD += KLD.item()

        loss.backward()
        optimizer.step()

    print("\tEpoch", epoch + 1, "complete!",
        "\tAverage Loss: ", overall_loss / (batch_idx*batch_size),
        "\tAverage reconst Loss: ", overall_reconst / (batch_idx*batch_size),
        "\tAverage KLD Loss: ", overall_KLD / (batch_idx*batch_size))

print("Finish!!")

Start training VAE...
	Epoch 1 complete! 	Average Loss:  61.301479783414806 	Average reconst Loss:  57.789486723923616 	Average KLD Loss:  3.5119932159740177
	Epoch 2 complete! 	Average Loss:  48.17732983745036 	Average reconst Loss:  40.928734248364734 	Average KLD Loss:  7.248595744949299
	Epoch 3 complete! 	Average Loss:  44.460303668500316 	Average reconst Loss:  35.19606549587937 	Average KLD Loss:  9.264238230739604
	Epoch 4 complete! 	Average Loss:  43.05884925018057 	Average reconst Loss:  32.79745586426965 	Average KLD Loss:  10.261393494222963
	Epoch 5 complete! 	Average Loss:  42.43756691042406 	Average reconst Loss:  31.679172119605575 	Average KLD Loss:  10.75839473269983
	Epoch 6 complete! 	Average Loss:  41.949635312828 	Average reconst Loss:  30.68709606477098 	Average KLD Loss:  11.262539150312007
	Epoch 7 complete! 	Average Loss:  41.63010383246678 	Average reconst Loss:  30.12286478396598 	Average KLD Loss:  11.507239069634858
	Epoch 8 complete! 	Average Loss:  41.47

In [11]:
def eval(model, dataloader):
    model.eval()
    cos = nn.CosineSimilarity(dim=1, eps=1e-6)
    cos_sim = 0.0
    mse = 0.0
    n_sample = 0.0
    l1 = 0.0
    with torch.no_grad():
        for batch_idx, dict in enumerate(dataloader):
            x = dict['embed']
            x = x.to(DEVICE)
            n_sample += x.shape[0]
            x_hat, _, _ = model(x)
            cos_sim += cos(x_hat, x).sum()
            mse += ((x-x_hat)**2).sum()
            l1 += abs(x-x_hat).sum()
    return {"cos_sim": cos_sim/n_sample, "mse": mse/n_sample, 'l1': l1/n_sample}


In [12]:
test_set = EmbedDataset('dataset/vctk/') 
test_loader = DataLoader(test_set, batch_size=batch_size, num_workers=8, drop_last=True)

Total number of training embeds is 44242.
Total number of training speakers is 109.


In [13]:
test_set1 = EmbedDataset('dataset/ESD/') 
test_loader1 = DataLoader(test_set, batch_size=batch_size, num_workers=8, drop_last=True)

Total number of training embeds is 35000.
Total number of training speakers is 20.


In [14]:
eval(model,train_loader)

{'cos_sim': tensor(0.8666, device='cuda:0'),
 'mse': tensor(0.8702, device='cuda:0'),
 'l1': tensor(9.5041, device='cuda:0')}

In [15]:
eval(model,test_loader)

{'cos_sim': tensor(0.8271, device='cuda:0'),
 'mse': tensor(0.9186, device='cuda:0'),
 'l1': tensor(9.7605, device='cuda:0')}

In [16]:
eval(model,test_loader1)

{'cos_sim': tensor(0.8272, device='cuda:0'),
 'mse': tensor(0.9181, device='cuda:0'),
 'l1': tensor(9.7604, device='cuda:0')}

In [16]:
with torch.no_grad():
    noise = torch.randn(32, 64).to(DEVICE)
    generated_emb = model.Decoder(noise)

cos_sim = cos(generated_emb, x[4:5, :])
cos_sim, cos_sim.max(), cos_sim.mean()

(tensor([0.6788, 0.6243, 0.6997, 0.6445, 0.5810, 0.5060, 0.6196, 0.7270, 0.6426,
         0.4953, 0.5370, 0.5346, 0.5735, 0.6005, 0.7828, 0.6194, 0.6329, 0.5573,
         0.5213, 0.6468, 0.7166, 0.6712, 0.5224, 0.6597, 0.6319, 0.5520, 0.5926,
         0.5480, 0.6194, 0.6254, 0.6834, 0.6536], device='cuda:0'),
 tensor(0.7828, device='cuda:0'),
 tensor(0.6157, device='cuda:0'))

In [17]:
torch.save(model.state_dict(), "psg_mylossf_50.pt")